# 01 Data Audit

This notebook performs the first reproducibility audit for **Robust Flood-Risk Prioritisation in India**.

The audit is intentionally non-destructive: it loads the raw CSV files, records observed structure and potential data-quality issues, and writes audit tables to `outputs/tables`. It does **not** remove, impute, standardise, rename, or correct any source values.

## Setup

Paths are defined with `pathlib`. The setup cell searches upward from the current working directory to find the repository root, so the notebook can be run from either the repository root or the `notebooks` directory.

In [4]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

EXPECTED_RAW_FILES = [
    "DFSI.csv",
    "District_FloodedArea.csv",
    "District_FloodImpact.csv",
    "India_Flood_Inventory_v3.csv",
]


def find_project_root(start_path):
    """Find the repository root by searching for the expected raw-data files."""
    start_path = Path(start_path).resolve()
    candidates = [start_path, *start_path.parents]
    for candidate in candidates:
        raw_dir = candidate / "data" / "raw"
        if all((raw_dir / file_name).exists() for file_name in EXPECTED_RAW_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from the repository root "
        "or from a subdirectory inside it. Expected to find data/raw with: "
        f"{EXPECTED_RAW_FILES}"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CSV_FILES = {
    "DFSI": RAW_DIR / "DFSI.csv",
    "District_FloodedArea": RAW_DIR / "District_FloodedArea.csv",
    "District_FloodImpact": RAW_DIR / "District_FloodImpact.csv",
    "India_Flood_Inventory_v3": RAW_DIR / "India_Flood_Inventory_v3.csv",
}

print(f"Project root: {PROJECT_ROOT}")

missing_files = [str(path) for path in CSV_FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing expected raw CSV file(s): {missing_files}")

Project root: /Users/abhijithshaji/Documents/india-flood-index-robustness


## Load Raw CSV Files

Each CSV is loaded exactly as a raw table. No values are cleaned, standardised, imputed, or removed.

In [5]:
datasets = {}

for dataset_name, csv_path in CSV_FILES.items():
    df = pd.read_csv(csv_path, low_memory=False)
    datasets[dataset_name] = df
    print(f"{dataset_name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

DFSI: 744 rows x 3 columns
District_FloodedArea: 732 rows x 4 columns
District_FloodImpact: 732 rows x 5 columns
India_Flood_Inventory_v3: 6,876 rows x 23 columns


## Column Names and Inferred Datatypes

The datatypes below are pandas-inferred datatypes from the raw CSV load. They are descriptive only and are not used to coerce source values.

In [6]:
schema_records = []

for dataset_name, df in datasets.items():
    print(f"\n{dataset_name}")
    schema = pd.DataFrame({
        "column_name": df.columns,
        "inferred_dtype": df.dtypes.astype(str).values,
    })
    display(schema)
    schema_records.extend(
        {
            "dataset": dataset_name,
            "file_name": CSV_FILES[dataset_name].name,
            "column_name": row.column_name,
            "inferred_dtype": row.inferred_dtype,
        }
        for row in schema.itertuples(index=False)
    )

schema_df = pd.DataFrame(schema_records)


DFSI


,column_name,inferred_dtype
0,Unnamed: 0,object
1,State_Name,object
2,DFSI,float64



District_FloodedArea


,column_name,inferred_dtype
0,Dist_Name,object
1,Percent_Flooded_Area,float64
2,Parmanent_Water,float64
3,Corrected_Percent_Flooded_Area,float64



District_FloodImpact


,column_name,inferred_dtype
0,Dist_Name,object
1,Human_fatality,int64
2,Human_injured,int64
3,Population,int64
4,Mean_Flood_Duration,float64



India_Flood_Inventory_v3


,column_name,inferred_dtype
0,Unnamed: 0,int64
1,UEI,object
2,Start Date,object
3,End Date,object
4,Duration(Days),float64
5,Main Cause,object
6,Location,float64
7,Districts,object
8,State,object
9,Latitude,float64


## First Five Rows

The previews below are included for human inspection of raw values, naming patterns, and obvious formatting issues.

In [7]:
for dataset_name, df in datasets.items():
    print(f"\n{dataset_name}")
    display(df.head())


DFSI


,Unnamed: 0,State_Name,DFSI
0,Patna,BIHAR,19.300564
1,Murshidabad,WEST BENGAL,18.910884
2,Thane,MAHARASHTRA,18.861789
3,Guntur,ANDHRA PRADESH,18.832069
4,North Twenty Four Pargan*,WEST BENGAL,18.770538



District_FloodedArea


,Dist_Name,Percent_Flooded_Area,Parmanent_Water,Corrected_Percent_Flooded_Area
0,Dadra & Nagar Haveli,3.614705,0.645537,2.969168
1,Daman,2.853534,1.308996,1.544538
2,Diu,5.799818,4.979982,0.819836
3,North & Middle Andaman,1.314817,0.000000,1.314817
4,South Andaman,1.443452,0.000000,1.443452



District_FloodImpact


,Dist_Name,Human_fatality,Human_injured,Population,Mean_Flood_Duration
0,Dadra & Nagar Haveli,10,0,618459,1.0
1,Daman,0,0,190125,NaN
2,Diu,0,0,67412,NaN
3,North & Middle Andaman,0,0,101677,1.0
4,South Andaman,1,0,257900,2.0



India_Flood_Inventory_v3


,Unnamed: 0,UEI,Start Date,End Date,Duration(Days),Main Cause,Location,Districts,State,Latitude,Longitude,Severity,Area Affected,Human fatality,Human injured,Human Displaced,Animal Fatality,Description of Casualties/injured,Extent of damage,Event Source,Event Souce ID,District_LGD_Codes,State_Codes
0,563,UEI-IMD-FL-1967-0001,02-07-1967 00:00,08-07-1967 00:00,7.0,flood,NaN,NaN,Assam,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IMD,NaN,NaN,18
1,564,UEI-IMD-FL-1967-0002,22-07-1967 00:00,28-07-1967 00:00,7.0,flood,NaN,NaN,"Maharashtra, Gujarat",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IMD,NaN,NaN,"27,24"
2,565,UEI-IMD-FL-1967-0003,01-08-1967 00:00,30-08-1967 00:00,30.0,flood,NaN,NaN,Uttar Pradesh,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IMD,NaN,NaN,9
3,566,UEI-IMD-FL-1967-0004,08-09-1967 00:00,09-09-1967 00:00,2.0,flood,NaN,"Bhadrak, Dhenkanal, Jajapur, Subarnapur, Nuapada, Balangir, Boudh, Cuttack, Kandhamal, Nayagarh, Khordha, Kalahandi,...","Odisha, Bihar, Uttar Pradesh, Madhya Pradesh, Rajasthan, Haryana",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Marooning of Barwer Express,IMD,NaN,"348, 352, 356, None, 368, 345, 349, 350, 359, 367, 362, 358, 355, 432, 484, 370, 363, 364, 344, 360, 354, 353, 365, ...","21,10,9,23,8,6"
4,567,UEI-IMD-FL-1968-0001,22-06-1968 00:00,28-06-1968 00:00,7.0,flood,NaN,NaN,Assam,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Affecting about one million people,an area of over 5,00,000 hectares including cropped area of 81,000 hectares",IMD,NaN,NaN,18


## Missing Values by Column

Missingness is calculated as both counts and percentages. This table is for audit only; no missing values are filled or dropped.

In [5]:
missingness_records = []

for dataset_name, df in datasets.items():
    row_count = len(df)
    missing_counts = df.isna().sum()
    for column_name, missing_count in missing_counts.items():
        missingness_records.append({
            "dataset": dataset_name,
            "file_name": CSV_FILES[dataset_name].name,
            "column_name": column_name,
            "inferred_dtype": str(df[column_name].dtype),
            "row_count": row_count,
            "missing_count": int(missing_count),
            "missing_percent": round((missing_count / row_count) * 100, 4) if row_count else np.nan,
            "non_missing_count": int(row_count - missing_count),
        })

missingness_summary = pd.DataFrame(missingness_records)
display(missingness_summary)

,dataset,file_name,column_name,inferred_dtype,row_count,missing_count,missing_percent,non_missing_count
0,DFSI,DFSI.csv,Unnamed: 0,object,744,0,0.0000,744
1,DFSI,DFSI.csv,State_Name,object,744,0,0.0000,744
2,DFSI,DFSI.csv,DFSI,float64,744,1,0.1344,743
3,District_FloodedArea,District_FloodedArea.csv,Dist_Name,object,732,0,0.0000,732
4,District_FloodedArea,District_FloodedArea.csv,Percent_Flooded_Area,float64,732,0,0.0000,732
5,District_FloodedArea,District_FloodedArea.csv,Parmanent_Water,float64,732,0,0.0000,732
6,District_FloodedArea,District_FloodedArea.csv,Corrected_Percent_Flooded_Area,float64,732,0,0.0000,732
7,District_FloodImpact,District_FloodImpact.csv,Dist_Name,object,732,0,0.0000,732
8,District_FloodImpact,District_FloodImpact.csv,Human_fatality,int64,732,0,0.0000,732
9,District_FloodImpact,District_FloodImpact.csv,Human_injured,int64,732,0,0.0000,732


## Duplicate Complete Rows

Complete-row duplicates are counted across all columns exactly as loaded.

In [6]:
complete_duplicate_records = []

for dataset_name, df in datasets.items():
    duplicate_mask = df.duplicated(keep=False)
    duplicate_count = int(duplicate_mask.sum())
    duplicate_groups = int(df.loc[duplicate_mask].drop_duplicates().shape[0]) if duplicate_count else 0
    complete_duplicate_records.append({
        "dataset": dataset_name,
        "file_name": CSV_FILES[dataset_name].name,
        "row_count": len(df),
        "complete_duplicate_row_count": duplicate_count,
        "complete_duplicate_row_percent": round((duplicate_count / len(df)) * 100, 4) if len(df) else np.nan,
        "complete_duplicate_group_count": duplicate_groups,
    })
    print(f"{dataset_name}: {duplicate_count:,} complete duplicate rows across {duplicate_groups:,} duplicate row pattern(s)")

complete_duplicate_summary = pd.DataFrame(complete_duplicate_records)
display(complete_duplicate_summary)

DFSI: 0 complete duplicate rows across 0 duplicate row pattern(s)
District_FloodedArea: 0 complete duplicate rows across 0 duplicate row pattern(s)
District_FloodImpact: 0 complete duplicate rows across 0 duplicate row pattern(s)
India_Flood_Inventory_v3: 0 complete duplicate rows across 0 duplicate row pattern(s)


,dataset,file_name,row_count,complete_duplicate_row_count,complete_duplicate_row_percent,complete_duplicate_group_count
0,DFSI,DFSI.csv,744,0,0.0,0
1,District_FloodedArea,District_FloodedArea.csv,732,0,0.0,0
2,District_FloodImpact,District_FloodImpact.csv,732,0,0.0,0
3,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,6876,0,0.0,0


## Repeated District and State Name Combinations

This section identifies repeated combinations where both district-like and state-like columns are available. It preserves the original spelling, spacing, and case of names. A normalised review key is also reported only to help humans notice near-duplicates caused by whitespace or casing; it is not used to alter the data.

In [7]:
def find_columns(columns, include_terms, exclude_terms=None):
    """Return columns whose names contain any include term and none of the exclude terms."""
    exclude_terms = exclude_terms or []
    matches = []
    for column in columns:
        lowered = column.lower()
        if any(term in lowered for term in include_terms) and not any(term in lowered for term in exclude_terms):
            matches.append(column)
    return matches


def review_key(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


duplicate_district_records = []

for dataset_name, df in datasets.items():
    district_columns = find_columns(df.columns, ["district", "dist"], ["lgd", "code"])
    state_columns = find_columns(df.columns, ["state"], ["lgd", "code"])
    if not district_columns:
        print(f"{dataset_name}: no district-like name column found")
        continue

    if state_columns:
        group_columns = [district_columns[0], state_columns[0]]
    else:
        group_columns = [district_columns[0]]

    grouped = (
        df.groupby(group_columns, dropna=False)
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
        .sort_values(["row_count"] + group_columns, ascending=[False] + [True] * len(group_columns))
    )

    for row in grouped.to_dict(orient="records"):
        district_value = row.get(district_columns[0], pd.NA)
        state_value = row.get(state_columns[0], pd.NA) if state_columns else pd.NA
        duplicate_district_records.append({
            "dataset": dataset_name,
            "file_name": CSV_FILES[dataset_name].name,
            "district_column": district_columns[0],
            "state_column": state_columns[0] if state_columns else pd.NA,
            "district_name": district_value,
            "state_name": state_value,
            "district_review_key": review_key(district_value),
            "state_review_key": review_key(state_value),
            "row_count": int(row["row_count"]),
            "audit_note": "Repeated district/state name combination" if state_columns else "Repeated district name; no state-like column available in this dataset",
        })

duplicate_district_names = pd.DataFrame(duplicate_district_records)
display(duplicate_district_names)

DFSI: no district-like name column found


,dataset,file_name,district_column,state_column,district_name,state_name,district_review_key,state_review_key,row_count,audit_note
0,District_FloodedArea,District_FloodedArea.csv,Dist_Name,<NA>,Aurangabad,<NA>,aurangabad,<NA>,2,Repeated district name; no state-like column available in this dataset
1,District_FloodedArea,District_FloodedArea.csv,Dist_Name,<NA>,Balrampur,<NA>,balrampur,<NA>,2,Repeated district name; no state-like column available in this dataset
2,District_FloodedArea,District_FloodedArea.csv,Dist_Name,<NA>,Bilaspur,<NA>,bilaspur,<NA>,2,Repeated district name; no state-like column available in this dataset
3,District_FloodedArea,District_FloodedArea.csv,Dist_Name,<NA>,Hamirpur,<NA>,hamirpur,<NA>,2,Repeated district name; no state-like column available in this dataset
4,District_FloodedArea,District_FloodedArea.csv,Dist_Name,<NA>,Pratapgarh,<NA>,pratapgarh,<NA>,2,Repeated district name; no state-like column available in this dataset
...,...,...,...,...,...,...,...,...,...,...
720,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,Districts,State,Yadadri Bhuvanagiri,Telangana,yadadri bhuvanagiri,telangana,2,Repeated district/state name combination
721,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,Districts,State,Yadgir,Karnataka,yadgir,karnataka,2,Repeated district/state name combination
722,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,Districts,State,Yamunanagar,Haryana,yamunanagar,haryana,2,Repeated district/state name combination
723,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,Districts,State,NaN,Bihar,<NA>,bihar,2,Repeated district/state name combination


## District and State Identifier Audit

This audit identifies district/state name fields, checks whether LGD district and state codes are available, and records missingness in those identifier columns. It also reports date ranges in the flood-event inventory.

In [8]:
def format_date(value):
    if pd.isna(value):
        return pd.NA
    return value.strftime("%Y-%m-%d")


identifier_records = []

for dataset_name, df in datasets.items():
    district_name_columns = find_columns(df.columns, ["district", "dist"], ["lgd", "code"])
    state_name_columns = find_columns(df.columns, ["state"], ["lgd", "code"])
    district_lgd_columns = [column for column in df.columns if "district" in column.lower() and "lgd" in column.lower()]
    state_lgd_columns = [column for column in df.columns if "state" in column.lower() and ("lgd" in column.lower() or "code" in column.lower())]

    identifier_groups = {
        "district_name": district_name_columns,
        "state_name": state_name_columns,
        "district_lgd_code": district_lgd_columns,
        "state_lgd_code": state_lgd_columns,
    }

    for identifier_type, columns in identifier_groups.items():
        if columns:
            for column in columns:
                missing_count = int(df[column].isna().sum())
                identifier_records.append({
                    "dataset": dataset_name,
                    "file_name": CSV_FILES[dataset_name].name,
                    "audit_type": "identifier_availability",
                    "identifier_type": identifier_type,
                    "column_name": column,
                    "available": True,
                    "row_count": len(df),
                    "missing_count": missing_count,
                    "missing_percent": round((missing_count / len(df)) * 100, 4) if len(df) else np.nan,
                    "distinct_non_missing_values": int(df[column].nunique(dropna=True)),
                    "audit_note": "Identifier column found; values preserved as loaded",
                })
        else:
            identifier_records.append({
                "dataset": dataset_name,
                "file_name": CSV_FILES[dataset_name].name,
                "audit_type": "identifier_availability",
                "identifier_type": identifier_type,
                "column_name": pd.NA,
                "available": False,
                "row_count": len(df),
                "missing_count": pd.NA,
                "missing_percent": pd.NA,
                "distinct_non_missing_values": pd.NA,
                "audit_note": "No matching identifier column found",
            })

inventory_name = "India_Flood_Inventory_v3"
inventory_df = datasets[inventory_name]
date_columns = [column for column in inventory_df.columns if "date" in column.lower()]

for column in date_columns:
    parsed_dates = pd.to_datetime(inventory_df[column], dayfirst=True, errors="coerce")
    identifier_records.append({
        "dataset": inventory_name,
        "file_name": CSV_FILES[inventory_name].name,
        "audit_type": "date_range",
        "identifier_type": "event_date",
        "column_name": column,
        "available": True,
        "row_count": len(inventory_df),
        "missing_count": int(inventory_df[column].isna().sum()),
        "missing_percent": round((inventory_df[column].isna().sum() / len(inventory_df)) * 100, 4) if len(inventory_df) else np.nan,
        "distinct_non_missing_values": int(inventory_df[column].nunique(dropna=True)),
        "parsed_min_date": format_date(parsed_dates.min()),
        "parsed_max_date": format_date(parsed_dates.max()),
        "unparseable_non_missing_count": int((parsed_dates.isna() & inventory_df[column].notna()).sum()),
        "audit_note": "Date range parsed for audit only; raw date strings are unchanged",
    })

if {"Start Date", "End Date"}.issubset(inventory_df.columns):
    start_dates = pd.to_datetime(inventory_df["Start Date"], dayfirst=True, errors="coerce")
    end_dates = pd.to_datetime(inventory_df["End Date"], dayfirst=True, errors="coerce")
    identifier_records.append({
        "dataset": inventory_name,
        "file_name": CSV_FILES[inventory_name].name,
        "audit_type": "date_interval_quality",
        "identifier_type": "event_date_interval",
        "column_name": "Start Date / End Date",
        "available": True,
        "row_count": len(inventory_df),
        "missing_count": int((start_dates.isna() | end_dates.isna()).sum()),
        "missing_percent": round(((start_dates.isna() | end_dates.isna()).sum() / len(inventory_df)) * 100, 4) if len(inventory_df) else np.nan,
        "distinct_non_missing_values": pd.NA,
        "parsed_min_date": format_date(start_dates.min()),
        "parsed_max_date": format_date(end_dates.max()),
        "unparseable_non_missing_count": int(((start_dates.isna() & inventory_df["Start Date"].notna()) | (end_dates.isna() & inventory_df["End Date"].notna())).sum()),
        "start_after_end_count": int((start_dates > end_dates).sum()),
        "audit_note": "Rows where parsed start date is after parsed end date are potential issues for review",
    })

district_identifier_audit = pd.DataFrame(identifier_records)
display(district_identifier_audit)

,dataset,file_name,audit_type,identifier_type,column_name,available,row_count,missing_count,missing_percent,distinct_non_missing_values,audit_note,parsed_min_date,parsed_max_date,unparseable_non_missing_count,start_after_end_count
0,DFSI,DFSI.csv,identifier_availability,district_name,<NA>,False,744,<NA>,<NA>,<NA>,No matching identifier column found,NaN,NaN,NaN,NaN
1,DFSI,DFSI.csv,identifier_availability,state_name,State_Name,True,744,0,0.0,37,Identifier column found; values preserved as loaded,NaN,NaN,NaN,NaN
2,DFSI,DFSI.csv,identifier_availability,district_lgd_code,<NA>,False,744,<NA>,<NA>,<NA>,No matching identifier column found,NaN,NaN,NaN,NaN
3,DFSI,DFSI.csv,identifier_availability,state_lgd_code,<NA>,False,744,<NA>,<NA>,<NA>,No matching identifier column found,NaN,NaN,NaN,NaN
4,District_FloodedArea,District_FloodedArea.csv,identifier_availability,district_name,Dist_Name,True,732,0,0.0,726,Identifier column found; values preserved as loaded,NaN,NaN,NaN,NaN
5,District_FloodedArea,District_FloodedArea.csv,identifier_availability,state_name,<NA>,False,732,<NA>,<NA>,<NA>,No matching identifier column found,NaN,NaN,NaN,NaN
6,District_FloodedArea,District_FloodedArea.csv,identifier_availability,district_lgd_code,<NA>,False,732,<NA>,<NA>,<NA>,No matching identifier column found,NaN,NaN,NaN,NaN
7,District_FloodedArea,District_FloodedArea.csv,identifier_availability,state_lgd_code,<NA>,False,732,<NA>,<NA>,<NA>,No matching identifier column found,NaN,NaN,NaN,NaN
8,District_FloodImpact,District_FloodImpact.csv,identifier_availability,district_name,Dist_Name,True,732,0,0.0,726,Identifier column found; values preserved as loaded,NaN,NaN,NaN,NaN
9,District_FloodImpact,District_FloodImpact.csv,identifier_availability,state_name,<NA>,False,732,<NA>,<NA>,<NA>,No matching identifier column found,NaN,NaN,NaN,NaN


## Potential Data-Quality Issues for Human Review

The records below collect audit flags only. They are intended to guide later review and do not modify the raw data.

In [9]:
issue_records = []

def add_issue(dataset_name, issue_type, column_name=pd.NA, affected_rows=pd.NA, details=""):
    issue_records.append({
        "dataset": dataset_name,
        "file_name": CSV_FILES[dataset_name].name,
        "issue_type": issue_type,
        "column_name": column_name,
        "affected_rows": affected_rows,
        "details": details,
        "review_status": "requires_human_review",
    })


for dataset_name, df in datasets.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        if missing_count:
            add_issue(dataset_name, "missing_values", column, missing_count, "Column contains missing values")

        if column.lower().startswith("unnamed"):
            add_issue(dataset_name, "unnamed_column", column, len(df), "Column name suggests a saved index or unnamed source field")

        if any(term in column.lower() for term in ["parmanent", "souce"]):
            add_issue(dataset_name, "possible_source_column_typo", column, len(df), "Column spelling appears unusual; preserve until reviewed")

        if pd.api.types.is_numeric_dtype(df[column]):
            negative_count = int((df[column] < 0).sum())
            if negative_count:
                add_issue(dataset_name, "negative_numeric_values", column, negative_count, "Negative values in numeric field may need domain review")

        if pd.api.types.is_object_dtype(df[column]):
            as_text = df[column].dropna().astype(str)
            leading_trailing = int(as_text.str.match(r"^\s|.*\s$").sum())
            repeated_spaces = int(as_text.str.contains(r"\s{2,}", regex=True).sum())
            if leading_trailing:
                add_issue(dataset_name, "leading_or_trailing_whitespace", column, leading_trailing, "Text values have leading or trailing whitespace")
            if repeated_spaces:
                add_issue(dataset_name, "repeated_internal_whitespace", column, repeated_spaces, "Text values contain repeated whitespace")

    duplicate_count = int(df.duplicated(keep=False).sum())
    if duplicate_count:
        add_issue(dataset_name, "complete_duplicate_rows", pd.NA, duplicate_count, "Complete rows are repeated exactly")

    district_name_columns = find_columns(df.columns, ["district", "dist"], ["lgd", "code"])
    state_name_columns = find_columns(df.columns, ["state"], ["lgd", "code"])
    if not district_name_columns:
        add_issue(dataset_name, "district_name_column_not_found", pd.NA, len(df), "No district-like name column detected")
    if not state_name_columns:
        add_issue(dataset_name, "state_name_column_not_found", pd.NA, len(df), "No state-like name column detected")

    district_lgd_columns = [column for column in df.columns if "district" in column.lower() and "lgd" in column.lower()]
    state_lgd_columns = [column for column in df.columns if "state" in column.lower() and ("lgd" in column.lower() or "code" in column.lower())]
    if not district_lgd_columns:
        add_issue(dataset_name, "district_lgd_code_column_not_found", pd.NA, len(df), "No LGD district code column detected")
    if not state_lgd_columns:
        add_issue(dataset_name, "state_lgd_code_column_not_found", pd.NA, len(df), "No LGD state code column detected")

if not duplicate_district_names.empty:
    for row in duplicate_district_names.itertuples(index=False):
        add_issue(row.dataset, "repeated_district_state_combination", row.district_column, int(row.row_count), row.audit_note)

if {"Start Date", "End Date"}.issubset(inventory_df.columns):
    start_dates = pd.to_datetime(inventory_df["Start Date"], dayfirst=True, errors="coerce")
    end_dates = pd.to_datetime(inventory_df["End Date"], dayfirst=True, errors="coerce")
    start_after_end = int((start_dates > end_dates).sum())
    if start_after_end:
        add_issue(inventory_name, "start_date_after_end_date", "Start Date / End Date", start_after_end, "Parsed start date is after parsed end date")

data_quality_issues = pd.DataFrame(issue_records)
display(data_quality_issues)

,dataset,file_name,issue_type,column_name,affected_rows,details,review_status
0,DFSI,DFSI.csv,unnamed_column,Unnamed: 0,744,Column name suggests a saved index or unnamed source field,requires_human_review
1,DFSI,DFSI.csv,leading_or_trailing_whitespace,Unnamed: 0,1,Text values have leading or trailing whitespace,requires_human_review
2,DFSI,DFSI.csv,repeated_internal_whitespace,Unnamed: 0,2,Text values contain repeated whitespace,requires_human_review
3,DFSI,DFSI.csv,missing_values,DFSI,1,Column contains missing values,requires_human_review
4,DFSI,DFSI.csv,district_name_column_not_found,<NA>,744,No district-like name column detected,requires_human_review
...,...,...,...,...,...,...,...
771,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,repeated_district_state_combination,Districts,2,Repeated district/state name combination,requires_human_review
772,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,repeated_district_state_combination,Districts,2,Repeated district/state name combination,requires_human_review
773,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,repeated_district_state_combination,Districts,2,Repeated district/state name combination,requires_human_review
774,India_Flood_Inventory_v3,India_Flood_Inventory_v3.csv,repeated_district_state_combination,Districts,2,Repeated district/state name combination,requires_human_review


## Save Audit Tables

The requested audit tables, plus a detailed data-quality issue log for human review, are saved to `outputs/tables`.

In [10]:
inventory_records = []

for dataset_name, df in datasets.items():
    dataset_issues = data_quality_issues.query("dataset == @dataset_name") if not data_quality_issues.empty else pd.DataFrame()
    inventory_records.append({
        "dataset": dataset_name,
        "file_name": CSV_FILES[dataset_name].name,
        "relative_path": CSV_FILES[dataset_name].relative_to(PROJECT_ROOT).as_posix(),
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": " | ".join(map(str, df.columns)),
        "inferred_dtypes": " | ".join(f"{column}: {dtype}" for column, dtype in df.dtypes.astype(str).items()),
        "complete_duplicate_row_count": int(df.duplicated(keep=False).sum()),
        "columns_with_missing_values": int(df.isna().any().sum()),
        "total_missing_values": int(df.isna().sum().sum()),
        "detected_issue_count": len(dataset_issues),
        "detected_issue_types": " | ".join(sorted(dataset_issues["issue_type"].dropna().unique())) if not dataset_issues.empty else "",
    })

data_inventory = pd.DataFrame(inventory_records)

data_inventory.to_csv(TABLE_DIR / "data_inventory.csv", index=False)
missingness_summary.to_csv(TABLE_DIR / "missingness_summary.csv", index=False)
duplicate_district_names.to_csv(TABLE_DIR / "duplicate_district_names.csv", index=False)
district_identifier_audit.to_csv(TABLE_DIR / "district_identifier_audit.csv", index=False)
data_quality_issues.to_csv(TABLE_DIR / "data_quality_issues.csv", index=False)

saved_tables = sorted(path.relative_to(PROJECT_ROOT).as_posix() for path in TABLE_DIR.glob("*.csv"))
print("Saved audit tables:")
for table in saved_tables:
    print(f"- {table}")

Saved audit tables:
- outputs/tables/data_inventory.csv
- outputs/tables/data_quality_issues.csv
- outputs/tables/district_identifier_audit.csv
- outputs/tables/duplicate_district_names.csv
- outputs/tables/missingness_summary.csv


## Notes

This notebook is an audit checkpoint. Any flagged issue should be reviewed before downstream modelling or prioritisation choices are made. Raw values remain unchanged.